# 🏥 MIRA — 04: Human-in-the-Loop & Streaming

**Goal of this notebook:**
1. Add a **Human-in-the-Loop (HITL) interrupt** before Agent 4 finalizes — a doctor reviews and approves/rejects
2. **Stream** agent outputs as they're generated (not just the final blob)
3. Persist graph state with a **checkpointer** so the interrupt can pause and resume
4. Wrap everything into a clean `mira_pipeline.py`-ready function for the Streamlit UI

```
         [Clinical Question]
                 ↓
   Agent 1 → Agent 2 → Agent 3
                 ↓
         🛑 INTERRUPT — pause here
         👨‍⚕️ Doctor reviews Agent 3's reasoning
         ✅ Approve → Agent 4 finalizes
         ❌ Reject  → feedback loops back to Agent 3
```

> **Why this matters clinically:** No AI-generated clinical recommendation should reach a patient chart without a human checkpoint. This is the difference between a toy demo and a responsible clinical AI system.

## Cell 1 — Install Dependencies

In [ ]:
# !pip install langchain langchain-openai langgraph openai faiss-cpu pandas --quiet

## Cell 2 — Imports & Config

In [ ]:
import os
import sqlite3
import pickle
import json
import uuid
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from typing import TypedDict
from openai import OpenAI

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ─── CONFIG ──────────────────────────────────────────────────────────────────
OPENAI_API_KEY = "your-key-here"
DB_PATH        = Path("./mira_data/mimic.db")
FAISS_PATH     = Path("./mira_data/medical_faiss.index")
META_PATH      = Path("./mira_data/faiss_metadata.pkl")
SCHEMA_PATH    = Path("./mira_data/db_schema.txt")
# ─────────────────────────────────────────────────────────────────────────────

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-4o", temperature=0, streaming=True)

print("✅ Imports done")

## Cell 3 — Reload Data Layer

In [ ]:
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
DB_SCHEMA = SCHEMA_PATH.read_text()
print("✅ SQLite connected")

index = faiss.read_index(str(FAISS_PATH))
with open(META_PATH, "rb") as f:
    MEDICAL_GUIDELINES = pickle.load(f)
print(f"✅ FAISS loaded: {index.ntotal} vectors")

def get_openai_embeddings(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([r.embedding for r in response.data], dtype=np.float32)

print("✅ Data layer ready")

## Cell 4 — Tools (same as notebooks 02 & 03)

In [ ]:
@tool
def sql_query(query: str) -> str:
    """
    Execute a SQL SELECT query against the MIMIC-IV patient database.
    Tables: patients, admissions, labevents, d_labitems, diagnoses_icd.
    Always JOIN d_labitems ON labevents.itemid = d_labitems.itemid for readable lab names.
    Filter abnormal labs with: WHERE labevents.flag = 'abnormal'
    Returns JSON string of results or error.
    """
    try:
        df = pd.read_sql_query(query, conn)
        if df.empty:
            return json.dumps({"result": "No data found."})
        return json.dumps({"rows": df.head(20).to_dict(orient="records")}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e), "hint": "Check table/column names against the schema."})


@tool
def vector_search(query: str, k: int = 3) -> str:
    """
    Search the medical knowledge base using semantic similarity.
    Returns top-k most relevant guideline chunks with source citations.
    """
    try:
        query_vec = get_openai_embeddings([query])
        distances, indices = index.search(query_vec, k)
        results = []
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            chunk = MEDICAL_GUIDELINES[idx].copy()
            chunk["rank"] = rank + 1
            chunk["relevance_score"] = round(1 / (1 + float(dist)), 4)
            results.append(chunk)
        return json.dumps({"guidelines": results}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


print("✅ Tools ready")

## Cell 5 — Updated State (adds HITL fields)

Two new fields vs. notebook 03:
- `human_feedback` — what the doctor typed when reviewing
- `human_decision` — "approve" or "reject"

In [ ]:
class MIRAState(TypedDict):
    clinical_question: str
    sql_query_used: str
    sql_result: str
    sql_retry_count: int
    sql_error: str
    search_query_used: str
    guidelines: str
    clinical_reasoning: str
    final_report: str
    safety_flags: list[str]
    approved: bool
    # ── NEW for HITL ──
    human_decision: str      # "approve" / "reject" / "" (pending)
    human_feedback: str      # doctor's comments if rejected


print("✅ MIRAState updated with HITL fields")

## Cell 6 — Agents 1, 2, 3 (same logic, unchanged from notebook 03)

In [ ]:
def agent1_sql_extractor(state: MIRAState) -> MIRAState:
    print("\n🤖 AGENT 1: SQL Data Extractor")
    retry_count = state.get("sql_retry_count", 0)
    previous_error = state.get("sql_error", "")

    error_context = f"\nYour previous SQL failed: {previous_error}\nFix it." if previous_error else ""

    system_prompt = f"""You are a medical SQL expert. Write a single valid SQLite SELECT query.
DATABASE SCHEMA:
{DB_SCHEMA}
RULES:
- Always JOIN d_labitems ON labevents.itemid = d_labitems.itemid
- Use WHERE labevents.flag = 'abnormal' for abnormal results
- Return ONLY raw SQL, no markdown
{error_context}"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Clinical question: {state['clinical_question']}")
    ])
    raw_sql = response.content.strip().replace("```sql", "").replace("```", "").strip()
    print(f"📝 SQL: {raw_sql}")

    result = sql_query.invoke({"query": raw_sql})
    parsed = json.loads(result)

    if "error" in parsed:
        print(f"❌ SQL Error: {parsed['error']}")
        return {**state, "sql_query_used": raw_sql, "sql_result": "",
                "sql_error": parsed["error"], "sql_retry_count": retry_count + 1}

    print(f"✅ {len(parsed.get('rows', []))} rows returned")
    return {**state, "sql_query_used": raw_sql, "sql_result": result,
            "sql_error": "", "sql_retry_count": retry_count}


def should_retry_sql(state: MIRAState) -> str:
    if state.get("sql_error") and state.get("sql_retry_count", 0) < 3:
        print(f"🔄 Retrying SQL...")
        return "retry"
    return "ok"


def agent2_semantic_crossref(state: MIRAState) -> MIRAState:
    print("\n🤖 AGENT 2: Semantic Cross-Ref")
    sql_context = state.get("sql_result", "") or "No patient data retrieved."

    response = llm.invoke([
        SystemMessage(content="Write a 1-2 sentence semantic search query for clinical guidelines based on the patient data. Return ONLY the query."),
        HumanMessage(content=f"Question: {state['clinical_question']}\nPatient data: {sql_context[:1000]}")
    ])
    search_query = response.content.strip()
    print(f"🔍 Query: {search_query}")

    guidelines_result = vector_search.invoke({"query": search_query, "k": 3})
    parsed = json.loads(guidelines_result)
    print(f"✅ {len(parsed.get('guidelines', []))} guideline chunks found")

    return {**state, "search_query_used": search_query, "guidelines": guidelines_result}


def agent3_clinical_reasoning(state: MIRAState) -> MIRAState:
    print("\n🤖 AGENT 3: Clinical Reasoning")
    sql_result = state.get("sql_result", "No patient data.")
    guidelines = state.get("guidelines", "No guidelines.")

    try:
        guideline_text = ""
        for g in json.loads(guidelines).get("guidelines", []):
            guideline_text += f"\n[{g['source']}] {g['topic']}:\n{g['text']}\n"
    except:
        guideline_text = guidelines

    # If this is a retry after human rejection, include feedback
    feedback_context = ""
    if state.get("human_decision") == "reject" and state.get("human_feedback"):
        feedback_context = f"\n\nIMPORTANT — A clinician reviewed your previous analysis and rejected it with this feedback:\n{state['human_feedback']}\nRevise your analysis accordingly."

    system_prompt = f"""You are an expert clinical AI assistant. Synthesize patient data + guidelines into:
## Patient Summary
## Identified Concerns
## Clinical Guideline Context
## Recommended Actions
Ground every claim in the data or a cited guideline. Never hallucinate values.{feedback_context}"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Question: {state['clinical_question']}\n\nPATIENT DATA:\n{sql_result[:2000]}\n\nGUIDELINES:\n{guideline_text[:2000]}")
    ])
    reasoning = response.content.strip()
    print(f"✅ Reasoning generated ({len(reasoning)} chars)")

    return {**state, "clinical_reasoning": reasoning, "human_decision": "", "human_feedback": ""}


print("✅ Agents 1, 2, 3 defined")

## Cell 7 — 🛑 The Human-in-the-Loop Node

This node does nothing by itself — its only job is to be a **pause point**. We use LangGraph's `interrupt_before` when compiling the graph, which stops execution right before this node runs, hands control back to us, and waits for human input.

**How it works in practice:**
1. Graph runs Agent 1 → 2 → 3, then **pauses** before `human_review`
2. We print Agent 3's reasoning to the doctor
3. Doctor types `approve` or `reject` (+ feedback)
4. We update the state and **resume** the graph with `.invoke(None, config)`

In [ ]:
def human_review_node(state: MIRAState) -> MIRAState:
    """
    This node is a no-op pass-through. The actual pausing happens via
    `interrupt_before=['human_review']` at graph compile time.
    By the time this node body runs, the human has already made a decision
    (we inject it into state before resuming).
    """
    print("\n🩺 HUMAN REVIEW NODE — decision received:", state.get("human_decision"))
    return state


def route_after_human_review(state: MIRAState) -> str:
    """
    If doctor rejected → loop back to Agent 3 to revise.
    If approved → proceed to Agent 4 for final safety check.
    """
    if state.get("human_decision") == "reject":
        print("🔄 Doctor rejected — sending back to Agent 3 for revision")
        return "revise"
    print("✅ Doctor approved — proceeding to final safety check")
    return "proceed"


print("✅ Human review node defined")

## Cell 8 — Agent 4: Critic & Safety (unchanged logic)

In [ ]:
def agent4_critic_safety(state: MIRAState) -> MIRAState:
    print("\n🤖 AGENT 4: Critic & Safety")
    reasoning  = state.get("clinical_reasoning", "")
    sql_result = state.get("sql_result", "")
    guidelines = state.get("guidelines", "")

    system_prompt = """You are a medical AI safety critic. Check for hallucinated values, 
guideline grounding, dangerous recommendations, and completeness.
Respond ONLY in JSON: {"approved": true/false, "safety_flags": [], "final_report": "..."}"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"PATIENT DATA:\n{sql_result[:1500]}\n\nGUIDELINES:\n{guidelines[:1000]}\n\nANALYSIS:\n{reasoning}")
    ])

    try:
        raw = response.content.strip().replace("```json", "").replace("```", "").strip()
        critic_output = json.loads(raw)
    except Exception:
        critic_output = {"approved": True, "safety_flags": [], "final_report": reasoning}

    approved     = critic_output.get("approved", True)
    safety_flags = critic_output.get("safety_flags", [])
    final_report = critic_output.get("final_report", reasoning)

    print(f"{'✅ APPROVED' if approved else '❌ NOT APPROVED'} | Flags: {safety_flags or 'None'}")

    return {**state, "final_report": final_report, "safety_flags": safety_flags, "approved": approved}


print("✅ Agent 4 defined")

## Cell 9 — Build the Graph with Checkpointer + Interrupt

Two new things vs. notebook 03:
1. **`MemorySaver`** — a checkpointer that persists state between the pause and resume
2. **`interrupt_before=["human_review"]`** — tells LangGraph to stop right before that node

In [ ]:
graph_builder = StateGraph(MIRAState)

graph_builder.add_node("sql_extractor",     agent1_sql_extractor)
graph_builder.add_node("semantic_crossref", agent2_semantic_crossref)
graph_builder.add_node("clinical_reasoning", agent3_clinical_reasoning)
graph_builder.add_node("human_review",      human_review_node)
graph_builder.add_node("critic_safety",     agent4_critic_safety)

graph_builder.set_entry_point("sql_extractor")

graph_builder.add_conditional_edges(
    "sql_extractor", should_retry_sql,
    {"retry": "sql_extractor", "ok": "semantic_crossref"}
)
graph_builder.add_edge("semantic_crossref", "clinical_reasoning")
graph_builder.add_edge("clinical_reasoning", "human_review")

# After human review: approve → critic, reject → back to Agent 3
graph_builder.add_conditional_edges(
    "human_review", route_after_human_review,
    {"proceed": "critic_safety", "revise": "clinical_reasoning"}
)
graph_builder.add_edge("critic_safety", END)

# Checkpointer — required for interrupt/resume to work
checkpointer = MemorySaver()

mira_graph_hitl = graph_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["human_review"]
)

print("✅ MIRA graph compiled with HITL interrupt")
print("   Pauses before: human_review")
print("   Resume path   : approve → critic_safety | reject → clinical_reasoning (revise)")

## Cell 10 — Run Until the Interrupt

Each run needs a unique `thread_id` — this is how the checkpointer knows which paused conversation to resume later.

In [ ]:
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

initial_state: MIRAState = {
    "clinical_question": "Find patients with abnormal creatinine. Do they show signs of Acute Kidney Injury based on guidelines?",
    "sql_query_used": "", "sql_result": "", "sql_retry_count": 0, "sql_error": "",
    "search_query_used": "", "guidelines": "", "clinical_reasoning": "",
    "final_report": "", "safety_flags": [], "approved": False,
    "human_decision": "", "human_feedback": ""
}

print(f"🧵 Thread ID: {thread_id}")
print("\n▶️  Running graph until interrupt...\n")

# Runs Agent 1 → 2 → 3, then STOPS before human_review
result = mira_graph_hitl.invoke(initial_state, config)

print("\n" + "="*60)
print("🛑 GRAPH PAUSED — awaiting human review")
print("="*60)
print("\n📋 Agent 3's Clinical Reasoning (pending your approval):\n")
print(result["clinical_reasoning"])

## Cell 11 — 👨‍⚕️ Doctor Reviews & Decides

In the notebook, simulate the doctor's decision by setting variables directly.
In the real Streamlit app (next step after this notebook), this becomes a text input + approve/reject buttons.

In [ ]:
# ─── SIMULATE DOCTOR INPUT — change these to test approve vs reject ──────────
doctor_decision = "approve"   # change to "reject" to test the revision loop
doctor_feedback = ""          # e.g. "Please also mention dialysis criteria" if rejecting
# ───────────────────────────────────────────────────────────────────────────────

print(f"👨‍⚕️ Doctor decision: {doctor_decision}")
if doctor_feedback:
    print(f"💬 Feedback: {doctor_feedback}")

# Update the checkpointed state with the human's decision
mira_graph_hitl.update_state(
    config,
    {"human_decision": doctor_decision, "human_feedback": doctor_feedback}
)

print("\n▶️  Resuming graph...\n")

# Resume execution — pass None to continue from where it paused
final_result = mira_graph_hitl.invoke(None, config)

print("\n✅ Graph finished")

## Cell 12 — Print Final Report

In [ ]:
print("="*60)
print("📋 MIRA FINAL REPORT (post human review)")
print("="*60)
print(f"\nHuman decision : {final_result.get('human_decision')}")
print(f"Critic approved: {final_result.get('approved')}")
print(f"Safety flags   : {final_result.get('safety_flags') or 'None'}")
print(f"\n{'-'*60}")
print(final_result.get("final_report", "No report."))

## Cell 13 — Test the REJECT Path

Run a fresh thread, but this time reject with feedback — watch Agent 3 revise its analysis.

In [ ]:
thread_id_2 = str(uuid.uuid4())
config_2 = {"configurable": {"thread_id": thread_id_2}}

initial_state_2: MIRAState = {
    "clinical_question": "Which patients have abnormal lab results? Summarize the concerns.",
    "sql_query_used": "", "sql_result": "", "sql_retry_count": 0, "sql_error": "",
    "search_query_used": "", "guidelines": "", "clinical_reasoning": "",
    "final_report": "", "safety_flags": [], "approved": False,
    "human_decision": "", "human_feedback": ""
}

result_2 = mira_graph_hitl.invoke(initial_state_2, config_2)
print("\n🛑 Paused. Agent 3's first draft:\n")
print(result_2["clinical_reasoning"][:500], "...\n")

# Reject with specific feedback
mira_graph_hitl.update_state(config_2, {
    "human_decision": "reject",
    "human_feedback": "Be more specific about exact lab values and units. Add urgency level for each finding."
})

print("🔄 Resuming with rejection feedback...\n")
final_result_2 = mira_graph_hitl.invoke(None, config_2)

print("\n✅ Revised report after doctor feedback:\n")
print(final_result_2["final_report"])

## Cell 14 — Streaming Agent Output (Token by Token)

Instead of waiting for Agent 3's full response, we can stream it as it's generated — important for the Streamlit UI so the doctor sees progress in real time.

In [ ]:
def stream_clinical_reasoning(question: str, sql_result: str, guidelines: str):
    """
    Streams Agent 3's reasoning token by token instead of returning it all at once.
    This is what powers a 'typing' effect in the Streamlit UI.
    """
    try:
        guideline_text = ""
        for g in json.loads(guidelines).get("guidelines", []):
            guideline_text += f"\n[{g['source']}] {g['topic']}:\n{g['text']}\n"
    except:
        guideline_text = guidelines

    system_prompt = """You are an expert clinical AI assistant. Synthesize patient data + guidelines into:
## Patient Summary
## Identified Concerns
## Clinical Guideline Context
## Recommended Actions
Ground every claim in the data or a cited guideline."""

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Question: {question}\n\nPATIENT DATA:\n{sql_result[:2000]}\n\nGUIDELINES:\n{guideline_text[:2000]}")
    ]

    full_response = ""
    for chunk in llm.stream(messages):
        token = chunk.content
        full_response += token
        print(token, end="", flush=True)  # in Streamlit this becomes st.write_stream()

    return full_response


# Test streaming with a fresh SQL + vector search
sql_result_sample = sql_query.invoke({
    "query": """SELECT p.subject_id, p.gender, p.anchor_age, d.label AS lab_name, l.value, l.flag
                FROM labevents l JOIN patients p ON l.subject_id = p.subject_id
                JOIN d_labitems d ON l.itemid = d.itemid
                WHERE l.flag = 'abnormal' LIMIT 5"""
})
guidelines_sample = vector_search.invoke({"query": "abnormal lab values clinical significance", "k": 2})

print("🌊 Streaming Agent 3's reasoning live:\n")
print("-"*60)
streamed_output = stream_clinical_reasoning(
    "Summarize the abnormal lab findings", sql_result_sample, guidelines_sample
)
print("\n" + "-"*60)
print("\n✅ Streaming complete")

## ✅ Summary

| Component | Status |
|---|---|
| `MemorySaver` checkpointer | ✅ |
| `interrupt_before=["human_review"]` | ✅ |
| Approve path → Agent 4 | ✅ |
| Reject path → revise via Agent 3 with feedback | ✅ |
| Token-by-token streaming | ✅ |
| Thread-based state persistence | ✅ |

**What's left for a production app:**
- Convert this notebook into `mira_pipeline.py` (functions only, no notebook output)
- Build a Streamlit front-end:
  - Text input for the clinical question
  - `st.write_stream()` for Agent 3's reasoning
  - Approve/Reject buttons + feedback textbox for the HITL step
  - Final report display with safety flags highlighted
- Swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` if you want persistence across app restarts

🎉 **This completes the full MIRA agent pipeline — from raw SQL/FAISS data to a safety-reviewed, human-approved clinical report.**